In [0]:
import json
import re
import time
import uuid
from datetime import datetime, timezone
from functools import reduce

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StructType, StringType

In [0]:
def _widget(name, default, choices=None):
    try:
        if choices:
            dbutils.widgets.dropdown(name, default, choices)
        else:
            dbutils.widgets.text(name, default)
        return dbutils.widgets.get(name)
    except Exception:
        return default


TARGET_ENV = _widget("target_env", "prod", ["prod", "dev"]).lower()
DRY_RUN = _widget("dry_run", "false", ["false", "true"]).lower() == "true"
TABLE_FILTER = _widget("table_filter", "").strip().lower()
BACKFILL_PENDING_LOOKUPS = (
    _widget("backfill_pending_lookups", "true", ["true", "false"]).lower() == "true"
)
MAX_CLASSIFICATION_ATTEMPTS = int(_widget("max_classification_attempts", "30"))
MATERIALIZE_MIN_ROWS = int(_widget("materialize_min_rows", "10000"))
VALIDATE_TARGET_UNIQUENESS = (
    _widget("validate_target_uniqueness", "false", ["false", "true"]).lower() == "true"
)

CATALOG = "4_prod" if TARGET_ENV == "prod" else "8_dev"
RAW_SCHEMA = "raw"
STAGING_SCHEMA = "staging"
TMP_SCHEMA = "tmp"
ABANDONED_SCHEMA = "staging_abandoned"
EXCLUDED_SCHEMA = "staging_excluded"
LOG_CATALOG = "6_mgmt"
LOG_SCHEMA = "logs"

RUN_ID = str(uuid.uuid4())
RUN_TOKEN = RUN_ID.replace("-", "")[:16]
RUN_STARTED_AT = datetime.now(timezone.utc)
SCRATCH_TABLES = []

TRUST_MAP_TBL = f"{CATALOG}.{TMP_SCHEMA}.org_to_trust_map"
POLICY_TBL = f"{CATALOG}.{TMP_SCHEMA}.trust_table_policy"
ENC_ORG_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_enc_org_map"
HUB_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_mapping_hub"
CHANGED_ENC_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_changed_encounters"
CONTROL_TBL = f"{CATALOG}.{TMP_SCHEMA}.incr_updt_trust_control"
FLAG_TBL = f"{LOG_CATALOG}.{LOG_SCHEMA}.organization_flags"
RUN_LOG_TBL = f"{LOG_CATALOG}.{LOG_SCHEMA}.trust_classification_run_log_v2"
TABLE_LOG_TBL = f"{LOG_CATALOG}.{LOG_SCHEMA}.trust_classification_table_log_v2"

CANONICAL_ORG_TRUST = {
    873843: "Barts", 8367658: "Barts", 669849: "Barts", 9073614: "Barts",
    2681833: "Barts", 4401825: "Barts", 3203824: "Barts", 2681830: "Barts",
    8061679: "Barts", 669848: "Barts", 8467812: "Barts", 2681824: "Barts",
    2619824: "Barts", 2681827: "Barts", 3203825: "Barts", 691988: "Barts",
    3125827: "Barts", 8061682: "Barts", 8061694: "Barts", 2641824: "Barts",
    2641827: "Barts", 669847: "Barts", 8056759: "Barts", 8061685: "Barts",
    2641830: "Barts", 3201824: "Barts", 691989: "Barts", 669845: "Barts",
    669843: "Barts", 8061691: "Barts", 669846: "Barts", 3199824: "Barts",
    669850: "Barts", 6333825: "Barts", 669844: "Barts", 8397458: "Barts",
    8152502: "Barts", 671843: "Barts", 613843: "Barts",
    9161976: "BHRUT", 9163579: "BHRUT", 9161983: "BHRUT", 723896: "BHRUT",
    9161987: "BHRUT", 9161988: "BHRUT", 9163583: "BHRUT", 9161989: "BHRUT",
}

# This is the maintenance notebook's former protection policy, now used by both
# notebooks through POLICY_TBL. BHRUT rows outside this set are archived to
# staging_excluded before removal from staging/raw.
KEEP_BHRUT_TABLES = {
    "mill_person", "mill_address", "mill_person_alias", "mill_person_name",
    "mill_person_patient", "mill_person_info", "mill_person_org_reltn",
    "mill_person_person_reltn", "mill_person_prsnl_reltn", "mill_prsnl",
    "mill_prsnl_alias", "mill_prsnl_org_reltn", "mill_prsnl_reltn",
}

# Used only when staging history is unavailable. The normal path discovers the
# exact key used by the upstream IncrUpdtV2 staging MERGE.
KEY_OVERRIDES = {
    "mill_order_detail": ["ORDER_ID", "ACTION_SEQUENCE", "DETAIL_SEQUENCE"],
    "mill_order_comment": ["ORDER_ID", "ACTION_SEQUENCE", "COMMENT_TYPE_CD"],
    "mill_order_ingredient": ["ORDER_ID", "ACTION_SEQUENCE", "COMP_SEQUENCE"],
    "mill_episode_encntr_reltn": ["EPISODE_ENCNTR_RELTN_ID"],
    "mill_sch_location": ["SCHEDULE_ID"],
    "mill_sn_surg_case_proc_doc": ["SURG_CASE_PROC_DOC_ID"],
    "mill_ce_blob": ["EVENT_ID", "BLOB_SEQ_NUM"],
}

LOOKUP_SPECS = {
    "SURG_CASE_ID": ("mill_surgical_case", "SURG_CASE_ID"),
    "SURG_CASE_PROC_ID": ("mill_surg_case_procedure", "SURG_CASE_PROC_ID"),
    "DCP_FORMS_ACTIVITY_ID": ("mill_dcp_forms_activity", "DCP_FORMS_ACTIVITY_ID"),
    "EPISODE_ID": ("mill_episode_encntr_reltn", "EPISODE_ID"),
    "ORDER_ID": ("mill_orders", "ORDER_ID"),
    "PROBLEM_ID": ("mill_problem", "PROBLEM_ID"),
    "SCH_EVENT_ID": ("mill_sch_event_patient", "SCH_EVENT_ID"),
    "SCHEDULE_ID": ("mill_sch_schedule", "SCHEDULE_ID"),
    "IM_STUDY_ID": ("mill_im_study", "IM_STUDY_ID"),
    "IM_ACQUIRED_STUDY_ID": ("mill_im_acquired_study", "IM_ACQUIRED_STUDY_ID"),
    "CV_PROC_ID": ("mill_cv_proc", "CV_PROC_ID"),
    "EVENT_ID": ("mill_clinical_event", "EVENT_ID"),
}

SOURCE_PRIORITY = [
    "mill_encounter",
    "mill_surgical_case",
    "mill_surg_case_procedure",
    "mill_dcp_forms_activity",
    "mill_episode_encntr_reltn",
    "mill_orders",
    "mill_problem",
    "mill_sch_event_patient",
    "mill_sch_schedule",
    "mill_cv_proc",
    "mill_im_study",
    "mill_im_acquired_study",
    "mill_clinical_event",
]

spark.sql(f"USE CATALOG {CATALOG}")

print("=" * 88)
print("TRUST CLASSIFICATION INCREMENTAL - UPDATED")
print(f"run_id={RUN_ID} env={TARGET_ENV} dry_run={DRY_RUN}")
print("=" * 88)

In [0]:
def table_exists(fqn):
    return spark.catalog.tableExists(fqn)


def table_columns(fqn):
    return spark.table(fqn).columns


def upper_map(columns):
    return {c.upper(): c for c in columns}


def sql_string(value):
    return str(value).replace("'", "''")


def latest_metrics(fqn, operation=None):
    rows = spark.sql(f"DESCRIBE HISTORY {fqn} LIMIT 20").collect()
    for row in rows:
        if operation is None or row["operation"] == operation:
            return dict(row["operationMetrics"] or {})
    return {}


def metric_int(metrics, name):
    try:
        return int((metrics or {}).get(name, 0) or 0)
    except (TypeError, ValueError):
        return 0


def best_order_columns(columns):
    cu = upper_map(columns)
    preferred = [
        "VALID_UNTIL_DT_TM", "VALID_FROM_DT_TM", "LAST_UTC_TS",
        "ADC_UPDT", "UPDT_DT_TM", "_STAGED_AT",
    ]
    result = [cu[c] for c in preferred if c in cu]
    return result


def key_condition(left_alias, right_alias, keys):
    conditions = [
        F.col(f"{left_alias}.{k}").eqNullSafe(F.col(f"{right_alias}.{k}"))
        for k in keys
    ]
    return reduce(lambda a, b: a & b, conditions)


def any_null_key(keys, alias=None):
    cols = [F.col(f"{alias}.{k}" if alias else k).isNull() for k in keys]
    return reduce(lambda a, b: a | b, cols)


def row_fingerprint(columns):
    normalized = [
        F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")).alias(c)
        for c in columns
    ]
    return F.sha2(F.to_json(F.struct(*normalized)), 256)


def actual_table_count(fqn):
    return spark.table(fqn).count()


def write_run_log(status, table_count=0, failed_count=0, notes=None):
    values = [(RUN_ID, RUN_STARTED_AT, datetime.now(timezone.utc), TARGET_ENV, status,
               int(table_count), int(failed_count), notes)]
    schema = "run_id string, started_at timestamp, ended_at timestamp, target_env string, status string, table_count long, failed_count long, notes string"
    spark.createDataFrame(values, schema=schema).write.mode("append").saveAsTable(RUN_LOG_TBL)


def write_table_log(
    table_name,
    status,
    merge_keys,
    staged_rows,
    resolved_keys=0,
    promoted_keys=0,
    excluded_keys=0,
    abandoned_keys=0,
    remaining_rows=0,
    error_message=None,
):
    values = [(
        RUN_ID, datetime.now(timezone.utc), table_name, status,
        json.dumps(merge_keys), int(staged_rows), int(resolved_keys),
        int(promoted_keys), int(excluded_keys), int(abandoned_keys),
        int(remaining_rows), error_message,
    )]
    schema = (
        "run_id string, logged_at timestamp, table_name string, status string, "
        "merge_keys string, staged_rows long, resolved_keys long, promoted_keys long, "
        "excluded_keys long, abandoned_keys long, remaining_rows long, error_message string"
    )
    spark.createDataFrame(values, schema=schema).write.mode("append").saveAsTable(TABLE_LOG_TBL)


def ensure_column(fqn, name, data_type):
    if name.upper() not in upper_map(table_columns(fqn)):
        spark.sql(f"ALTER TABLE {fqn} ADD COLUMNS ({name} {data_type})")


def ensure_setup():
    for schema in [TMP_SCHEMA, STAGING_SCHEMA, ABANDONED_SCHEMA, EXCLUDED_SCHEMA]:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LOG_CATALOG}.{LOG_SCHEMA}")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TRUST_MAP_TBL} (
            organization_id BIGINT,
            trust STRING
        ) USING DELTA
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {POLICY_TBL} (
            table_name STRING,
            keep_bhrut BOOLEAN,
            updated_at TIMESTAMP
        ) USING DELTA
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {ENC_ORG_TBL} (
            ENCNTR_ID BIGINT,
            ORGANIZATION_ID BIGINT,
            updated_at TIMESTAMP
        ) USING DELTA
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {HUB_TBL} (
            key_type STRING,
            key_id BIGINT,
            ENCNTR_ID BIGINT,
            updated_at TIMESTAMP
        ) USING DELTA
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CHANGED_ENC_TBL} (
            ENCNTR_ID BIGINT,
            change_type STRING,
            detected_at TIMESTAMP
        ) USING DELTA
    """)
    for name, typ in [
        ("old_organization_id", "BIGINT"),
        ("new_organization_id", "BIGINT"),
        ("run_id", "STRING"),
        ("processed_at", "TIMESTAMP"),
    ]:
        ensure_column(CHANGED_ENC_TBL, name, typ)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CONTROL_TBL} (
            table_name STRING,
            checkpoint_key STRING,
            last_version BIGINT,
            last_timestamp TIMESTAMP,
            updated_at TIMESTAMP
        ) USING DELTA
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {FLAG_TBL} (
            organization_id BIGINT,
            organization_name STRING,
            flag_type STRING,
            flag_reason STRING,
            first_seen TIMESTAMP,
            last_seen TIMESTAMP,
            resolved BOOLEAN,
            resolved_at TIMESTAMP
        ) USING DELTA
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {RUN_LOG_TBL} (
            run_id STRING,
            started_at TIMESTAMP,
            ended_at TIMESTAMP,
            target_env STRING,
            status STRING,
            table_count BIGINT,
            failed_count BIGINT,
            notes STRING
        ) USING DELTA
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TABLE_LOG_TBL} (
            run_id STRING,
            logged_at TIMESTAMP,
            table_name STRING,
            status STRING,
            merge_keys STRING,
            staged_rows BIGINT,
            resolved_keys BIGINT,
            promoted_keys BIGINT,
            excluded_keys BIGINT,
            abandoned_keys BIGINT,
            remaining_rows BIGINT,
            error_message STRING
        ) USING DELTA
    """)

    trust_rows = [(int(k), v) for k, v in CANONICAL_ORG_TRUST.items()]
    trust_df = spark.createDataFrame(trust_rows, "organization_id long, trust string")
    DeltaTable.forName(spark, TRUST_MAP_TBL).alias("t").merge(
        trust_df.alias("s"),
        F.col("t.organization_id") == F.col("s.organization_id"),
    ).whenMatchedUpdate(set={"trust": F.col("s.trust")}).whenNotMatchedInsertAll().whenNotMatchedBySourceDelete().execute()

    all_policy_tables = sorted(KEEP_BHRUT_TABLES)
    policy_df = spark.createDataFrame(
        [(t, True, datetime.now(timezone.utc)) for t in all_policy_tables],
        "table_name string, keep_bhrut boolean, updated_at timestamp",
    )
    DeltaTable.forName(spark, POLICY_TBL).alias("t").merge(
        policy_df.alias("s"),
        F.col("t.table_name") == F.col("s.table_name"),
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().whenNotMatchedBySourceDelete().execute()


def discover_merge_keys(table_name, staging_fqn):
    cols = table_columns(staging_fqn)
    cu = upper_map(cols)

    try:
        history = spark.sql(f"DESCRIBE HISTORY {staging_fqn} LIMIT 200")
        predicates = (
            history.filter(F.col("operation") == "MERGE")
            .orderBy(F.col("version").desc())
            .select(F.col("operationParameters").getItem("predicate").alias("predicate"))
            .collect()
        )
        for row in predicates:
            predicate = row["predicate"] or ""
            names = re.findall(r"([A-Za-z_][A-Za-z0-9_]*)#[0-9]+L?", predicate)
            keys = []
            for name in names:
                upper = name.upper()
                if upper in cu and upper not in keys:
                    keys.append(upper)
            if keys:
                return [cu[k] for k in keys]
    except Exception as exc:
        print(f"  Key-history warning for {table_name}: {type(exc).__name__}: {exc}")

    override = KEY_OVERRIDES.get(table_name.lower())
    if override and all(k.upper() in cu for k in override):
        return [cu[k.upper()] for k in override]

    raise ValueError(
        f"No authoritative merge key found for {table_name}. "
        "Add it to KEY_OVERRIDES; heuristic keys are intentionally disabled."
    )


def ensure_raw_target(table_name, staging_fqn, raw_fqn):
    staging_cols = table_columns(staging_fqn)
    business_cols = [
        c for c in staging_cols
        if c.upper() not in {"_STAGED_AT", "_CLASSIFICATION_ATTEMPTS"}
    ]

    if not table_exists(raw_fqn):
        if DRY_RUN:
            print(f"  DRY RUN: would create {raw_fqn}")
            return business_cols
        spark.table(staging_fqn).select(*business_cols).limit(0).write.format("delta").saveAsTable(raw_fqn)

    raw_cols = table_columns(raw_fqn)
    raw_upper = upper_map(raw_cols)
    staging_types = {f.name.upper(): f.dataType.simpleString() for f in spark.table(staging_fqn).schema.fields}

    for enrichment in ["TRUST", "ENCNTR_ID", "ORGANIZATION_ID"]:
        if enrichment in upper_map(staging_cols) and enrichment not in raw_upper:
            if DRY_RUN:
                print(f"  DRY RUN: would add {enrichment} to {raw_fqn}")
            else:
                spark.sql(f"ALTER TABLE {raw_fqn} ADD COLUMNS ({enrichment} {staging_types[enrichment]})")

    raw_cols = table_columns(raw_fqn)
    raw_upper = upper_map(raw_cols)
    missing_business = [
        c for c in business_cols
        if c.upper() not in raw_upper and c.upper() not in {"TRUST", "ENCNTR_ID", "ORGANIZATION_ID"}
    ]
    if missing_business:
        raise ValueError(f"Raw target {raw_fqn} is missing business columns: {missing_business}")
    return raw_cols


def merge_unique(
    target_fqn,
    source_df,
    keys,
    update_columns=None,
    known_nonempty=False,
    source_is_unique=False,
):
    if not known_nonempty and source_df.limit(1).count() == 0:
        return {"source_rows": 0}

    merge_source = source_df
    if not source_is_unique:
        order_columns = best_order_columns(source_df.columns)
        if order_columns:
            source_window = Window.partitionBy(*keys).orderBy(
                *[F.col(column).desc_nulls_last() for column in order_columns]
            )
            merge_source = (
                source_df.withColumn("_merge_source_rn", F.row_number().over(source_window))
                .filter(F.col("_merge_source_rn") == 1)
                .drop("_merge_source_rn")
            )
        else:
            merge_source = source_df.dropDuplicates(keys)

    # This audit is deliberately optional. Running it for every table performs a
    # second full target scan before the Delta MERGE and was the dominant cost on
    # large raw tables and on the shared mapping hub. The standalone duplicate
    # repair notebook certifies/repairs legacy collisions before normal runs.
    if VALIDATE_TARGET_UNIQUENESS and table_exists(target_fqn):
        source_keys = merge_source.select(*keys).dropDuplicates()
        target_duplicates = (
            spark.table(target_fqn)
            .join(source_keys, keys, "inner")
            .groupBy(*keys)
            .count()
            .filter(F.col("count") > 1)
            .limit(1)
            .count()
        )
        if target_duplicates:
            raise ValueError(
                f"Target {target_fqn} already contains duplicate rows on {keys}. "
                "Repair or rebuild those keys before classification is resumed."
            )

    if DRY_RUN:
        return {"source_rows": merge_source.count(), "dry_run": True}

    target_cols = table_columns(target_fqn)
    target_upper = upper_map(target_cols)
    source_upper = upper_map(merge_source.columns)
    common = [target_upper[u] for u in target_upper if u in source_upper]
    update_cols = update_columns or [c for c in common if c.upper() not in {k.upper() for k in keys}]
    update_map = {c: F.col(f"s.{source_upper[c.upper()]}") for c in update_cols}
    insert_map = {c: F.col(f"s.{source_upper[c.upper()]}") for c in common}

    DeltaTable.forName(spark, target_fqn).alias("t").merge(
        merge_source.alias("s"),
        key_condition("t", "s", keys),
    ).whenMatchedUpdate(set=update_map).whenNotMatchedInsert(values=insert_map).execute()
    return latest_metrics(target_fqn, "MERGE")


def append_encounter_changes(changes_df):
    if changes_df.limit(1).count() == 0 or DRY_RUN:
        return
    out = (
        changes_df.select(
            F.col("ENCNTR_ID").cast("long"),
            F.lit("organization_remap").alias("change_type"),
            F.current_timestamp().alias("detected_at"),
            F.col("old_organization_id").cast("long"),
            F.col("new_organization_id").cast("long"),
            F.lit(RUN_ID).alias("run_id"),
            F.lit(None).cast("timestamp").alias("processed_at"),
        )
        .dropDuplicates(["ENCNTR_ID", "old_organization_id", "new_organization_id", "run_id"])
    )
    out.write.mode("append").saveAsTable(CHANGED_ENC_TBL)


def merge_encounter_map(source_df, track_changes=True):
    source = (
        source_df.select(
            F.col("ENCNTR_ID").cast("long").alias("ENCNTR_ID"),
            F.col("ORGANIZATION_ID").cast("long").alias("ORGANIZATION_ID"),
        )
        .filter((F.col("ENCNTR_ID").isNotNull()) & (F.col("ENCNTR_ID") != 0))
        .dropDuplicates(["ENCNTR_ID"])
    )
    if source.limit(1).count() == 0:
        return

    existing = spark.table(ENC_ORG_TBL).select(
        "ENCNTR_ID", F.col("ORGANIZATION_ID").alias("old_organization_id")
    )
    compared = source.join(existing, "ENCNTR_ID", "left")
    changes = compared.filter(
        F.col("old_organization_id").isNotNull()
        & ~F.col("old_organization_id").eqNullSafe(F.col("ORGANIZATION_ID"))
    ).select(
        "ENCNTR_ID",
        "old_organization_id",
        F.col("ORGANIZATION_ID").alias("new_organization_id"),
    )
    if track_changes:
        append_encounter_changes(changes)

    source = source.withColumn("updated_at", F.current_timestamp())
    merge_unique(
        ENC_ORG_TBL,
        source,
        ["ENCNTR_ID"],
        ["ORGANIZATION_ID", "updated_at"],
        known_nonempty=True,
        source_is_unique=True,
    )


def merge_hub(source_df, key_type):
    source = (
        source_df.select(
            F.lit(key_type).alias("key_type"),
            F.col("key_id").cast("long").alias("key_id"),
            F.col("ENCNTR_ID").cast("long").alias("ENCNTR_ID"),
        )
        .filter(
            F.col("key_id").isNotNull()
            & (F.col("key_id") != 0)
            & F.col("ENCNTR_ID").isNotNull()
            & (F.col("ENCNTR_ID") != 0)
        )
        .dropDuplicates(["key_type", "key_id"])
        .withColumn("updated_at", F.current_timestamp())
    )
    merge_unique(
        HUB_TBL,
        source,
        ["key_type", "key_id"],
        ["ENCNTR_ID", "updated_at"],
        source_is_unique=True,
    )


def raw_lookup_mapping(key_type, needed_keys):
    source_table, source_key = LOOKUP_SPECS[key_type]
    source_fqn = f"{CATALOG}.{RAW_SCHEMA}.{source_table}"
    if not table_exists(source_fqn):
        return None

    needed = needed_keys.select(F.col(key_type).alias("needed_key")).dropDuplicates()
    original_source_columns = table_columns(source_fqn)
    src = spark.table(source_fqn).alias("src").join(
        needed.alias("n"),
        F.col(f"src.{source_key}") == F.col("n.needed_key"),
        "inner",
    )

    if key_type == "SURG_CASE_PROC_ID":
        sc = spark.table(f"{CATALOG}.{RAW_SCHEMA}.mill_surgical_case").select(
            "SURG_CASE_ID", "ENCNTR_ID"
        )
        src = src.join(sc, "SURG_CASE_ID", "left")
        enc_expr = F.col("ENCNTR_ID")
    elif key_type == "IM_STUDY_ID":
        cp = spark.table(f"{CATALOG}.{RAW_SCHEMA}.mill_cv_proc").select(
            F.col("CV_PROC_ID").alias("_CV_PROC_ID"),
            F.col("ENCNTR_ID").alias("_CP_ENCNTR_ID"),
        )
        src = src.join(cp, F.col("ORIG_ENTITY_ID") == F.col("_CV_PROC_ID"), "left")
        enc_expr = F.coalesce(F.col("src.ENCNTR_ID"), F.col("_CP_ENCNTR_ID"))
    elif key_type == "IM_ACQUIRED_STUDY_ID":
        ims = spark.table(f"{CATALOG}.{RAW_SCHEMA}.mill_im_study").select(
            F.col("IM_STUDY_ID").alias("_IMS_STUDY_ID"),
            F.col("ENCNTR_ID").alias("_IMS_ENCNTR_ID"),
            F.col("ORIG_ENTITY_ID").alias("_IMS_ORIG_ENTITY_ID"),
        )
        cp = spark.table(f"{CATALOG}.{RAW_SCHEMA}.mill_cv_proc").select(
            F.col("CV_PROC_ID").alias("_CV_PROC_ID"),
            F.col("ENCNTR_ID").alias("_CP_ENCNTR_ID"),
        )
        src = src.join(
            ims,
            F.col("src.MATCHED_STUDY_ID") == F.col("_IMS_STUDY_ID"),
            "left",
        )
        src = src.join(
            cp,
            F.col("_IMS_ORIG_ENTITY_ID") == F.col("_CV_PROC_ID"),
            "left",
        )
        enc_expr = F.coalesce(F.col("_IMS_ENCNTR_ID"), F.col("_CP_ENCNTR_ID"))
    elif key_type == "PROBLEM_ID":
        enc_expr = F.coalesce(F.col("src.ORIGINATING_ENCNTR_ID"), F.col("src.UPDATE_ENCNTR_ID"))
    else:
        enc_expr = F.col("src.ENCNTR_ID")

    order_cols = best_order_columns(original_source_columns)
    ordering = [F.col(f"src.{c}").desc_nulls_last() for c in order_cols] or [
        F.col(f"src.{source_key}").desc()
    ]
    window = Window.partitionBy(F.col(f"src.{source_key}")).orderBy(*ordering)
    return (
        src.select(
            F.col(f"src.{source_key}").alias("key_id"),
            enc_expr.alias("ENCNTR_ID"),
            F.row_number().over(window).alias("_rn"),
        )
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )


def applicable_lookup_types(table_name, columns):
    cu = upper_map(columns)
    has_direct_route = "ENCNTR_ID" in cu or "ORGANIZATION_ID" in cu
    result = []
    for key_type, (source_table, _) in LOOKUP_SPECS.items():
        if key_type not in cu:
            continue
        # A source table's own hub entry is derived from that same table. Joining
        # it back to itself is redundant when the row already carries encounter
        # or organization data, and it caused large avoidable hub scans.
        if has_direct_route and source_table.lower() == table_name.lower():
            continue
        result.append(key_type)
    return result


def backfill_pending_lookups(table_name, staging_fqn, columns):
    if not BACKFILL_PENDING_LOOKUPS:
        return
    cu = upper_map(columns)
    staging = spark.table(staging_fqn)

    if "ENCNTR_ID" in cu and table_exists(f"{CATALOG}.{RAW_SCHEMA}.mill_encounter"):
        needed = staging.select(F.col(cu["ENCNTR_ID"]).alias("ENCNTR_ID")).filter(
            F.col("ENCNTR_ID").isNotNull() & (F.col("ENCNTR_ID") != 0)
        ).dropDuplicates()
        current = spark.table(ENC_ORG_TBL).select("ENCNTR_ID")
        missing = needed.join(current, "ENCNTR_ID", "left_anti")
        if missing.limit(1).count() > 0:
            encounters = spark.table(f"{CATALOG}.{RAW_SCHEMA}.mill_encounter").select(
                F.col("ENCNTR_ID").cast("long").alias("ENCNTR_ID"),
                F.col("ORGANIZATION_ID").cast("long").alias("ORGANIZATION_ID"),
            )
            merge_encounter_map(missing.join(encounters, "ENCNTR_ID", "inner"), track_changes=False)

    for key_type in applicable_lookup_types(table_name, columns):
        needed = staging.select(F.col(cu[key_type]).alias(key_type)).filter(
            F.col(key_type).isNotNull() & (F.col(key_type) != 0)
        ).dropDuplicates()
        current = spark.table(HUB_TBL).filter(F.col("key_type") == key_type).select(
            F.col("key_id").alias(key_type)
        )
        missing = needed.join(current, key_type, "left_anti")
        if missing.limit(1).count() == 0:
            continue
        mapping = raw_lookup_mapping(key_type, missing)
        if mapping is not None:
            merge_hub(mapping, key_type)


def current_version(fqn):
    return int(spark.sql(f"DESCRIBE HISTORY {fqn} LIMIT 1").first()["version"])


def get_checkpoint(fqn, checkpoint_key):
    rows = spark.sql(f"""
        SELECT last_version
        FROM {CONTROL_TBL}
        WHERE table_name = '{sql_string(fqn)}'
          AND checkpoint_key = '{sql_string(checkpoint_key)}'
        ORDER BY updated_at DESC
        LIMIT 1
    """).collect()
    return rows[0]["last_version"] if rows else None


def set_checkpoint(fqn, checkpoint_key, version):
    if DRY_RUN:
        return
    source = spark.createDataFrame(
        [(fqn, checkpoint_key, int(version), datetime.now(timezone.utc), datetime.now(timezone.utc))],
        "table_name string, checkpoint_key string, last_version long, last_timestamp timestamp, updated_at timestamp",
    )
    DeltaTable.forName(spark, CONTROL_TBL).alias("t").merge(
        source.alias("s"),
        (F.col("t.table_name") == F.col("s.table_name"))
        & (F.col("t.checkpoint_key") == F.col("s.checkpoint_key")),
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def refresh_encounters_from_cdf():
    fqn = f"{CATALOG}.{RAW_SCHEMA}.mill_encounter"
    if not table_exists(fqn):
        return
    end_version = current_version(fqn)
    last_version = get_checkpoint(fqn, "hub_encounter_v2")
    if last_version is None:
        set_checkpoint(fqn, "hub_encounter_v2", end_version)
        print("Encounter checkpoint initialized; pending staging keys will be backfilled on demand.")
        return
    if last_version >= end_version:
        return

    try:
        changes = (
            spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", int(last_version) + 1)
            .option("endingVersion", end_version)
            .table(fqn)
            .filter(F.col("_change_type").isin("insert", "update_postimage", "delete"))
        )
    except Exception as exc:
        print(
            f"Encounter CDF unavailable for versions {int(last_version) + 1}..{end_version}; "
            f"pending staging keys will be backfilled on demand: {type(exc).__name__}: {exc}"
        )
        return
    window = Window.partitionBy("ENCNTR_ID").orderBy(
        F.col("_commit_version").desc(),
        F.when(F.col("_change_type") == "delete", 2).otherwise(1).desc(),
    )
    latest = changes.withColumn("_rn", F.row_number().over(window)).filter(F.col("_rn") == 1)

    deleted = latest.filter(F.col("_change_type") == "delete").select(
        F.col("ENCNTR_ID").cast("long").alias("ENCNTR_ID")
    )
    if deleted.limit(1).count() > 0 and not DRY_RUN:
        old = spark.table(ENC_ORG_TBL).join(deleted, "ENCNTR_ID", "inner").select(
            "ENCNTR_ID",
            F.col("ORGANIZATION_ID").alias("old_organization_id"),
            F.lit(None).cast("long").alias("new_organization_id"),
        )
        append_encounter_changes(old)
        DeltaTable.forName(spark, ENC_ORG_TBL).alias("t").merge(
            deleted.alias("s"), F.col("t.ENCNTR_ID") == F.col("s.ENCNTR_ID")
        ).whenMatchedDelete().execute()

    upserts = latest.filter(F.col("_change_type") != "delete").select(
        "ENCNTR_ID", "ORGANIZATION_ID"
    )
    merge_encounter_map(upserts, track_changes=True)
    set_checkpoint(fqn, "hub_encounter_v2", end_version)


def refresh_hub_from_cdf():
    for key_type, (table_name, source_key) in LOOKUP_SPECS.items():
        fqn = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"
        if not table_exists(fqn):
            continue
        checkpoint_key = f"hub_{key_type}_v2"
        end_version = current_version(fqn)
        last_version = get_checkpoint(fqn, checkpoint_key)
        if last_version is None:
            set_checkpoint(fqn, checkpoint_key, end_version)
            continue
        if last_version >= end_version:
            continue

        try:
            changes = (
                spark.read.format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", int(last_version) + 1)
                .option("endingVersion", end_version)
                .table(fqn)
                .filter(F.col("_change_type").isin("insert", "update_postimage", "delete"))
            )
        except Exception as exc:
            print(
                f"{key_type} CDF unavailable for versions "
                f"{int(last_version) + 1}..{end_version}; demand backfill remains enabled: "
                f"{type(exc).__name__}: {exc}"
            )
            continue
        window = Window.partitionBy(source_key).orderBy(
            F.col("_commit_version").desc(),
            F.when(F.col("_change_type") == "delete", 2).otherwise(1).desc(),
        )
        latest = changes.withColumn("_rn", F.row_number().over(window)).filter(F.col("_rn") == 1)

        deleted = latest.filter(F.col("_change_type") == "delete").select(
            F.col(source_key).cast("long").alias("key_id")
        )
        if deleted.limit(1).count() > 0 and not DRY_RUN:
            DeltaTable.forName(spark, HUB_TBL).alias("t").merge(
                deleted.alias("s"),
                (F.col("t.key_type") == F.lit(key_type))
                & (F.col("t.key_id") == F.col("s.key_id")),
            ).whenMatchedDelete().execute()

        changed_keys = latest.filter(F.col("_change_type") != "delete").select(
            F.col(source_key).alias(key_type)
        ).dropDuplicates()
        if changed_keys.limit(1).count() > 0:
            mapping = raw_lookup_mapping(key_type, changed_keys)
            if mapping is not None:
                merge_hub(mapping, key_type)
        set_checkpoint(fqn, checkpoint_key, end_version)


def build_resolved(table_name, staging_fqn, keys):
    staging = spark.table(staging_fqn)
    metadata = {"_STAGED_AT", "_CLASSIFICATION_ATTEMPTS"}
    columns = staging.columns
    cu = upper_map(columns)

    valid = staging.filter(~any_null_key(keys))
    ordering = [F.col(c).desc_nulls_last() for c in best_order_columns(columns)]
    if not ordering:
        ordering = [F.col(k).desc_nulls_last() for k in keys]
    window = Window.partitionBy(*keys).orderBy(*ordering)
    resolved = valid.withColumn("_key_rn", F.row_number().over(window)).filter(
        F.col("_key_rn") == 1
    ).drop("_key_rn")

    trust_map = spark.table(TRUST_MAP_TBL)
    trust_map_join = trust_map.select(
        F.col("organization_id").alias("_tm_organization_id"),
        F.col("trust").alias("_tm_trust"),
    )
    enc_trust = (
        spark.table(ENC_ORG_TBL).alias("eo")
        .join(
            F.broadcast(trust_map_join.alias("tm")),
            F.col("eo.ORGANIZATION_ID") == F.col("tm._tm_organization_id"),
            "left",
        )
        .select(
            F.col("eo.ENCNTR_ID").alias("ENCNTR_ID"),
            F.col("eo.ORGANIZATION_ID").alias("ORGANIZATION_ID"),
            F.col("tm._tm_trust").alias("trust"),
        )
    )

    original_enc = (
        F.when(
            F.col(cu["ENCNTR_ID"]).isNotNull() & (F.col(cu["ENCNTR_ID"]) != 0),
            F.col(cu["ENCNTR_ID"]),
        )
        if "ENCNTR_ID" in cu
        else F.lit(None).cast("long")
    )
    original_org = (
        F.when(
            F.col(cu["ORGANIZATION_ID"]).isNotNull()
            & (F.col(cu["ORGANIZATION_ID"]) != 0),
            F.col(cu["ORGANIZATION_ID"]),
        )
        if "ORGANIZATION_ID" in cu
        else F.lit(None).cast("long")
    )
    route_candidates = []

    if "ORGANIZATION_ID" in cu:
        direct_org = trust_map.select(
            F.col("organization_id").alias("_direct_org_id"),
            F.col("trust").alias("_direct_org_trust"),
        )
        resolved = resolved.join(
            F.broadcast(direct_org),
            original_org == F.col("_direct_org_id"),
            "left",
        )
        route_candidates.append(
            (F.col("_direct_org_trust"), original_enc, original_org)
        )

    if "ENCNTR_ID" in cu:
        direct_enc = enc_trust.select(
            F.col("ENCNTR_ID").alias("_direct_enc_id"),
            F.col("ORGANIZATION_ID").alias("_direct_enc_org"),
            F.col("trust").alias("_direct_enc_trust"),
        )
        resolved = resolved.join(
            direct_enc,
            original_enc == F.col("_direct_enc_id"),
            "left",
        )
        route_candidates.append(
            (
                F.col("_direct_enc_trust"),
                F.col("_direct_enc_id"),
                F.col("_direct_enc_org"),
            )
        )

    for index, key_type in enumerate(applicable_lookup_types(table_name, columns)):
        lookup = (
            spark.table(HUB_TBL)
            .filter(F.col("key_type") == key_type)
            .join(enc_trust, "ENCNTR_ID", "left")
            .select(
                F.col("key_id").alias(f"_hub_key_{index}"),
                F.col("ENCNTR_ID").alias(f"_hub_enc_{index}"),
                F.col("ORGANIZATION_ID").alias(f"_hub_org_{index}"),
                F.col("trust").alias(f"_hub_trust_{index}"),
            )
        )
        resolved = resolved.join(
            lookup,
            F.col(cu[key_type]) == F.col(f"_hub_key_{index}"),
            "left",
        )
        route_candidates.append(
            (
                F.col(f"_hub_trust_{index}"),
                F.col(f"_hub_enc_{index}"),
                F.col(f"_hub_org_{index}"),
            )
        )

    trust_candidates = [candidate[0] for candidate in route_candidates]
    routed_encounters = [
        F.when(trust.isNotNull(), encounter)
        for trust, encounter, _ in route_candidates
    ]
    routed_organizations = [
        F.when(trust.isNotNull(), organization)
        for trust, _, organization in route_candidates
    ]
    resolved = (
        resolved.withColumn(
            "_resolved_encntr_id",
            F.coalesce(*(routed_encounters + [original_enc])),
        )
        .withColumn(
            "_resolved_organization_id",
            F.coalesce(*(routed_organizations + [original_org])),
        )
        .withColumn(
            "_resolved_trust",
            F.coalesce(*trust_candidates)
            if trust_candidates
            else F.lit(None).cast("string"),
        )
    )
    return resolved.select(
        *columns,
        "_resolved_encntr_id",
        "_resolved_organization_id",
        "_resolved_trust",
    )


def ensure_archive_table(target_fqn, source_df):
    if not table_exists(target_fqn):
        source_df.limit(0).write.format("delta").saveAsTable(target_fqn)
        return
    target_upper = upper_map(table_columns(target_fqn))
    for field in source_df.schema.fields:
        if field.name.upper() not in target_upper:
            spark.sql(
                f"ALTER TABLE {target_fqn} ADD COLUMNS "
                f"({field.name} {field.dataType.simpleString()})"
            )


def archive_rows(df, staging_columns, target_fqn, reason, expected_count=None):
    if expected_count is not None:
        if expected_count <= 0:
            return 0
    elif df.limit(1).count() == 0:
        return 0
    archived = (
        df.select(*staging_columns)
        .withColumn("_row_fingerprint", row_fingerprint(staging_columns))
        .withColumn("_archive_reason", F.lit(reason))
        .withColumn("_archived_at", F.current_timestamp())
        .withColumn("_run_id", F.lit(RUN_ID))
        .dropDuplicates(["_row_fingerprint"])
    )
    count = int(expected_count) if expected_count is not None else archived.count()
    if DRY_RUN:
        return count
    ensure_archive_table(target_fqn, archived)
    target_upper = upper_map(table_columns(target_fqn))
    source_upper = upper_map(archived.columns)
    insert_values = {
        target_upper[u]: F.col(f"s.{source_upper[u]}")
        for u in target_upper
        if u in source_upper
    }
    DeltaTable.forName(spark, target_fqn).alias("t").merge(
        archived.alias("s"),
        F.col("t._row_fingerprint") == F.col("s._row_fingerprint"),
    ).whenNotMatchedInsert(values=insert_values).execute()
    return count


def delete_by_fingerprint(staging_fqn, rows_df, staging_columns, known_nonempty=False):
    if DRY_RUN:
        return 0
    if not known_nonempty and rows_df.limit(1).count() == 0:
        return 0
    fingerprints = rows_df.select(
        row_fingerprint(staging_columns).alias("_row_fingerprint")
    ).dropDuplicates()
    target_normalized = [
        F.coalesce(F.col(f"t.{c}").cast("string"), F.lit("<NULL>")).alias(c)
        for c in staging_columns
    ]
    target_fingerprint = F.sha2(F.to_json(F.struct(*target_normalized)), 256)
    DeltaTable.forName(spark, staging_fqn).alias("t").merge(
        fingerprints.alias("s"),
        target_fingerprint == F.col("s._row_fingerprint"),
    ).whenMatchedDelete().execute()
    return metric_int(latest_metrics(staging_fqn, "MERGE"), "numTargetRowsDeleted")


def delete_by_keys(staging_fqn, keys_df, keys, known_nonempty=False):
    if DRY_RUN:
        return 0
    if not known_nonempty and keys_df.limit(1).count() == 0:
        return 0
    DeltaTable.forName(spark, staging_fqn).alias("t").merge(
        keys_df.select(*keys).dropDuplicates().alias("s"),
        key_condition("t", "s", keys),
    ).whenMatchedDelete().execute()
    return metric_int(latest_metrics(staging_fqn, "MERGE"), "numTargetRowsDeleted")


def increment_attempts_by_keys(staging_fqn, keys_df, keys, known_nonempty=False):
    if DRY_RUN:
        return
    if not known_nonempty and keys_df.limit(1).count() == 0:
        return
    DeltaTable.forName(spark, staging_fqn).alias("t").merge(
        keys_df.select(*keys).dropDuplicates().alias("s"),
        key_condition("t", "s", keys),
    ).whenMatchedUpdate(
        set={
            "_classification_attempts": F.coalesce(
                F.col("t._classification_attempts"), F.lit(0)
            ) + F.lit(1)
        }
    ).execute()


def prepare_raw_source(resolved_df, raw_fqn, keys, raw_cols=None):
    raw_cols = raw_cols or table_columns(raw_fqn)
    source_upper = upper_map(resolved_df.columns)
    expressions = []
    for raw_col in raw_cols:
        upper = raw_col.upper()
        if upper == "TRUST":
            expressions.append(F.col("_resolved_trust").alias(raw_col))
        elif upper == "ENCNTR_ID":
            original = (
                F.when(
                    F.col(source_upper["ENCNTR_ID"]).isNotNull()
                    & (F.col(source_upper["ENCNTR_ID"]) != 0),
                    F.col(source_upper["ENCNTR_ID"]),
                )
                if "ENCNTR_ID" in source_upper
                else F.lit(None).cast("long")
            )
            expressions.append(
                F.coalesce(F.col("_resolved_encntr_id"), original).alias(raw_col)
            )
        elif upper == "ORGANIZATION_ID":
            original = (
                F.when(
                    F.col(source_upper["ORGANIZATION_ID"]).isNotNull()
                    & (F.col(source_upper["ORGANIZATION_ID"]) != 0),
                    F.col(source_upper["ORGANIZATION_ID"]),
                )
                if "ORGANIZATION_ID" in source_upper
                else F.lit(None).cast("long")
            )
            expressions.append(
                F.coalesce(F.col("_resolved_organization_id"), original).alias(raw_col)
            )
        elif upper in source_upper:
            expressions.append(F.col(source_upper[upper]).alias(raw_col))
    source = resolved_df.select(*expressions)
    source_upper = upper_map(source.columns)
    missing_keys = [k for k in keys if k.upper() not in source_upper]
    if missing_keys:
        raise ValueError(f"Raw merge source lacks keys {missing_keys}")
    return source


def classify_table(table_name, staging_fqn, raw_fqn):
    started = time.time()
    staging_columns = table_columns(staging_fqn)
    keys = discover_merge_keys(table_name, staging_fqn)
    staging = spark.table(staging_fqn)
    null_key_condition = any_null_key(keys)
    attempts_col = F.coalesce(F.col("_classification_attempts"), F.lit(0))

    # One staging pass replaces the former full count, null-key count, null-at-
    # cap count, and valid-row existence probes.
    staging_stats = staging.agg(
        F.count(F.lit(1)).alias("staged_rows"),
        F.sum(F.when(null_key_condition, 1).otherwise(0)).alias("null_key_rows"),
        F.sum(
            F.when(
                null_key_condition & (attempts_col >= MAX_CLASSIFICATION_ATTEMPTS - 1),
                1,
            ).otherwise(0)
        ).alias("null_key_rows_at_cap"),
    ).first()
    staged_rows = int(staging_stats["staged_rows"] or 0)
    if staged_rows == 0:
        return

    null_key_count = int(staging_stats["null_key_rows"] or 0)
    null_cap_count = int(staging_stats["null_key_rows_at_cap"] or 0)
    valid_rows = staged_rows - null_key_count
    print(f"\n{table_name}: rows={staged_rows:,} keys={keys}")

    raw_columns = ensure_raw_target(table_name, staging_fqn, raw_fqn)
    backfill_pending_lookups(table_name, staging_fqn, staging_columns)

    deleted_rows = 0
    if null_key_count:
        null_key_df = staging.filter(null_key_condition)
        null_at_cap = null_key_df.filter(
            attempts_col >= MAX_CLASSIFICATION_ATTEMPTS - 1
        )
        if null_cap_count:
            archive_fqn = f"{CATALOG}.{ABANDONED_SCHEMA}.{table_name}"
            archive_rows(
                null_at_cap,
                staging_columns,
                archive_fqn,
                "null_merge_key",
                expected_count=null_cap_count,
            )
            deleted_rows += delete_by_fingerprint(
                staging_fqn,
                null_at_cap,
                staging_columns,
                known_nonempty=True,
            )
        if not DRY_RUN:
            DeltaTable.forName(spark, staging_fqn).update(
                condition=any_null_key(keys),
                set={
                    "_classification_attempts": F.coalesce(
                        F.col("_classification_attempts"), F.lit(0)
                    ) + F.lit(1)
                },
            )

    if valid_rows == 0:
        remaining = staged_rows if DRY_RUN else max(0, staged_rows - deleted_rows)
        write_table_log(
            table_name,
            "DRY_RUN" if DRY_RUN else "SUCCESS",
            keys,
            staged_rows,
            abandoned_keys=null_cap_count,
            remaining_rows=remaining,
        )
        print(f"  completed in {time.time() - started:.1f}s")
        return

    scratch_fqn = None
    try:
        resolved_plan = build_resolved(table_name, staging_fqn, keys)
        use_scratch = not DRY_RUN and valid_rows >= MATERIALIZE_MIN_ROWS
        if use_scratch:
            scratch_fqn = (
                f"{CATALOG}.{TMP_SCHEMA}._trust_resolved_v2_"
                f"{re.sub(r'[^A-Za-z0-9_]', '_', table_name)}_{RUN_TOKEN}"
            )
            SCRATCH_TABLES.append(scratch_fqn)
            spark.sql(f"DROP TABLE IF EXISTS {scratch_fqn}")
            (
                resolved_plan.write.format("delta")
                .mode("overwrite")
                .saveAsTable(scratch_fqn)
            )
            resolved = spark.table(scratch_fqn)
        else:
            resolved = resolved_plan

        keep_bhrut = table_name.lower() in KEEP_BHRUT_TABLES
        trust_upper = F.upper(F.col("_resolved_trust"))
        promoted_condition = F.col("_resolved_trust").isNotNull() & (
            (trust_upper != "BHRUT") | F.lit(keep_bhrut)
        )
        excluded_condition = (trust_upper == "BHRUT") & F.lit(not keep_bhrut)
        unresolved_condition = F.col("_resolved_trust").isNull()
        abandon_condition = unresolved_condition & (
            attempts_col >= MAX_CLASSIFICATION_ATTEMPTS - 1
        )
        retry_condition = unresolved_condition & (
            attempts_col < MAX_CLASSIFICATION_ATTEMPTS - 1
        )

        # One resolved-table aggregation replaces four separate count actions.
        outcome_stats = resolved.agg(
            F.count(F.lit(1)).alias("resolved_keys"),
            F.sum(F.when(promoted_condition, 1).otherwise(0)).alias("promoted_keys"),
            F.sum(F.when(excluded_condition, 1).otherwise(0)).alias("excluded_keys"),
            F.sum(F.when(abandon_condition, 1).otherwise(0)).alias("abandoned_keys"),
            F.sum(F.when(retry_condition, 1).otherwise(0)).alias("retry_keys"),
        ).first()
        resolved_keys = int(outcome_stats["resolved_keys"] or 0)
        promoted_keys = int(outcome_stats["promoted_keys"] or 0)
        excluded_keys = int(outcome_stats["excluded_keys"] or 0)
        unresolved_abandoned_keys = int(outcome_stats["abandoned_keys"] or 0)
        retry_keys = int(outcome_stats["retry_keys"] or 0)

        promoted = resolved.filter(promoted_condition)
        excluded = resolved.filter(excluded_condition)
        to_abandon = resolved.filter(abandon_condition)
        to_retry = resolved.filter(retry_condition)

        if promoted_keys:
            raw_source = prepare_raw_source(promoted, raw_fqn, keys, raw_columns)
            merge_unique(
                raw_fqn,
                raw_source,
                keys,
                known_nonempty=True,
                source_is_unique=True,
            )
            deleted_rows += delete_by_keys(
                staging_fqn, promoted, keys, known_nonempty=True
            )

        if excluded_keys:
            excluded_fqn = f"{CATALOG}.{EXCLUDED_SCHEMA}.{table_name}"
            archive_rows(
                excluded,
                staging_columns,
                excluded_fqn,
                "BHRUT_FILTERED",
                expected_count=excluded_keys,
            )
            deleted_rows += delete_by_keys(
                staging_fqn, excluded, keys, known_nonempty=True
            )

        if unresolved_abandoned_keys:
            abandon_fqn = f"{CATALOG}.{ABANDONED_SCHEMA}.{table_name}"
            archive_rows(
                to_abandon,
                staging_columns,
                abandon_fqn,
                "max_attempts_reached",
                expected_count=unresolved_abandoned_keys,
            )
            deleted_rows += delete_by_keys(
                staging_fqn, to_abandon, keys, known_nonempty=True
            )

        if retry_keys:
            increment_attempts_by_keys(
                staging_fqn, to_retry, keys, known_nonempty=True
            )

        abandoned_keys = null_cap_count + unresolved_abandoned_keys
        remaining = staged_rows if DRY_RUN else max(0, staged_rows - deleted_rows)
        write_table_log(
            table_name=table_name,
            status="DRY_RUN" if DRY_RUN else "SUCCESS",
            merge_keys=keys,
            staged_rows=staged_rows,
            resolved_keys=resolved_keys,
            promoted_keys=promoted_keys,
            excluded_keys=excluded_keys,
            abandoned_keys=abandoned_keys,
            remaining_rows=remaining,
        )
        print(
            f"  completed in {time.time() - started:.1f}s "
            f"(materialized={use_scratch})"
        )
    finally:
        # Do not retain every table's materialization for the duration of the run.
        # The previous behavior accumulated >100M rows of scratch data and made
        # later hub merges progressively slower.
        if scratch_fqn and not DRY_RUN:
            try:
                spark.sql(f"DROP TABLE IF EXISTS {scratch_fqn}")
            finally:
                if scratch_fqn in SCRATCH_TABLES:
                    SCRATCH_TABLES.remove(scratch_fqn)


def discover_staging_tables():
    rows = spark.sql(f"SHOW TABLES IN {CATALOG}.{STAGING_SCHEMA}").collect()
    names = sorted(
        row["tableName"] for row in rows
        if row["tableName"].lower().startswith("mill_")
    )
    if TABLE_FILTER:
        requested = {x.strip().lower() for x in TABLE_FILTER.split(",") if x.strip()}
        names = [name for name in names if name.lower() in requested]

    priority = {name: index for index, name in enumerate(SOURCE_PRIORITY)}
    return sorted(names, key=lambda name: (priority.get(name.lower(), 9999), name.lower()))


def flag_unknown_organizations():
    enc_orgs = (
        spark.table(ENC_ORG_TBL)
        .select(F.col("ORGANIZATION_ID").alias("_enc_org_id"))
        .filter(F.col("_enc_org_id").isNotNull() & (F.col("_enc_org_id") != 0))
        .dropDuplicates()
    )
    mapped_orgs = spark.table(TRUST_MAP_TBL).select(
        F.col("organization_id").alias("_mapped_org_id")
    )
    unknown = (
        enc_orgs.join(
            F.broadcast(mapped_orgs),
            F.col("_enc_org_id") == F.col("_mapped_org_id"),
            "left_anti",
        )
        .select(F.col("_enc_org_id").alias("ORGANIZATION_ID"))
    )
    if unknown.limit(1).count() == 0 or DRY_RUN:
        return

    org_fqn = f"{CATALOG}.{RAW_SCHEMA}.mill_organization"
    if table_exists(org_fqn) and "ORG_NAME" in upper_map(table_columns(org_fqn)):
        details = unknown.join(
            spark.table(org_fqn).select("ORGANIZATION_ID", "ORG_NAME"),
            "ORGANIZATION_ID",
            "left",
        ).select(
            F.col("ORGANIZATION_ID").alias("organization_id"),
            F.coalesce(F.col("ORG_NAME"), F.lit("UNKNOWN")).alias("organization_name"),
        )
    else:
        details = unknown.select(
            F.col("ORGANIZATION_ID").alias("organization_id"),
            F.lit("UNKNOWN").alias("organization_name"),
        )

    source = details.select(
        "organization_id",
        "organization_name",
        F.lit("UNKNOWN_TRUST").alias("flag_type"),
        F.lit("Organization is absent from the canonical trust map").alias("flag_reason"),
        F.current_timestamp().alias("first_seen"),
        F.current_timestamp().alias("last_seen"),
        F.lit(False).alias("resolved"),
        F.lit(None).cast("timestamp").alias("resolved_at"),
    )
    DeltaTable.forName(spark, FLAG_TBL).alias("t").merge(
        source.alias("s"),
        (F.col("t.organization_id") == F.col("s.organization_id"))
        & (F.col("t.flag_type") == F.col("s.flag_type"))
        & (F.col("t.resolved") == F.lit(False)),
    ).whenMatchedUpdate(
        set={
            "organization_name": F.col("s.organization_name"),
            "last_seen": F.col("s.last_seen"),
        }
    ).whenNotMatchedInsertAll().execute()


In [0]:
errors = []
tables = []

try:
    ensure_setup()
    refresh_encounters_from_cdf()
    refresh_hub_from_cdf()

    tables = discover_staging_tables()
    for table_name in tables:
        staging_fqn = f"{CATALOG}.{STAGING_SCHEMA}.{table_name}"
        raw_fqn = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"
        try:
            classify_table(table_name, staging_fqn, raw_fqn)
        except Exception as exc:
            message = f"{type(exc).__name__}: {str(exc)}"
            errors.append((table_name, message))
            try:
                error_staged_rows = actual_table_count(staging_fqn)
                write_table_log(
                    table_name=table_name,
                    status="FAILED",
                    merge_keys=[],
                    staged_rows=error_staged_rows,
                    remaining_rows=error_staged_rows,
                    error_message=message[:8000],
                )
            except Exception:
                pass
            print(f"FAILED {table_name}: {message}")

    flag_unknown_organizations()
    status = "FAILED" if errors else ("DRY_RUN" if DRY_RUN else "SUCCESS")
    write_run_log(
        status=status,
        table_count=len(tables),
        failed_count=len(errors),
        notes=json.dumps(errors)[:16000] if errors else None,
    )
finally:
    if not DRY_RUN:
        for scratch_fqn in SCRATCH_TABLES:
            try:
                spark.sql(f"DROP TABLE IF EXISTS {scratch_fqn}")
            except Exception:
                pass
    print("=" * 88)
    print(f"run_id={RUN_ID} tables={len(tables)} failures={len(errors)} dry_run={DRY_RUN}")
    print("=" * 88)

if errors:
    raise RuntimeError(
        f"Trust classification failed for {len(errors)} table(s): "
        + "; ".join(f"{table}: {message}" for table, message in errors[:10])
    )

result = {
    "run_id": RUN_ID,
    "status": "DRY_RUN" if DRY_RUN else "SUCCESS",
    "tables": len(tables),
    "failures": 0,
}
try:
    dbutils.notebook.exit(json.dumps(result))
except Exception:
    print(json.dumps(result))